In [1]:
import os
import pandas as pd

from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (Hemolytik 2.0, published 2026 version)

This notebook curates the **published 2026 version of Hemolytik 2.0** from the complete CSV dataset. It prepares both classification and quantitative hemolytic-activity datasets while keeping modified peptide records separate from the primary non-modified datasets.

- **Toxic effect / endpoint:** hemolytic
- **Quantitative endpoints:** HC50, LC50, LD50, and MHC
- **Source:** Hemolytik 2.0 (Singh et al., 2026)
- **Raw file:** `Hemolytik2_complete_data.csv`
- **Sequence scope:** non-modified peptides are retained in the primary datasets; modified peptides are exported separately.

The pipeline performs the following steps:

- loads and standardizes the complete Hemolytik 2.0 dataset;
- identifies non-modified peptides using terminal state, topology, stereochemistry, and non-natural residue annotations;
- preserves modified peptides with an explicit modification description;
- assigns binary hemolytic labels for classification;
- checks duplicate non-modified sequences and conflicting labels;
- preserves distinct modified peptide entities rather than collapsing them only by amino-acid sequence;
- parses HC50, LC50, LD50, and MHC measurements independently from the classification deduplication step;
- reports activity records that cannot be parsed into endpoint, value, and unit;
- builds task-specific metadata;
- exports classification and regression datasets, QC files, and metadata.

In [2]:
name_source = "Hemolytik2.0_2026"
name_task_classification = "toxic_effect_classification"
name_task_regression = "toxic_effect_regression"

activity_pattern = r"MHC|LD50|HC50|LC50"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants.
# The value of name_source must also match the corresponding entry in
# raw_data_description.xlsx.

- Reading and standardizing raw data

In [3]:
df_data_raw = pd.read_csv(f"{PATH_INPUT}/{name_source}/Hemolytik2_complete_data.csv")
number_of_raw_sequences = len(df_data_raw)
df_data_raw["sequence"] = (
    df_data_raw["seq"]
    .astype("string")
    .str.strip()
    .str.upper()
)
df_data_raw.shape

(13215, 21)

- Separating non-modified and modified peptides

In [4]:
non_nat_missing = ~df_data_raw["non_nat"].apply(has_annotation)
mask_no_mod = (
    df_data_raw["nter"].eq("Free")
    & df_data_raw["cter"].eq("Free")
    & df_data_raw["lyn_cyc"].eq("Linear")
    & df_data_raw["ldmix"].eq("L")
    & non_nat_missing
)

df_non_modified = (df_data_raw.loc[mask_no_mod, 
    ["sequence", "source", "non_hem", "activity"],]
    .copy().reset_index(drop=True)
)

df_modified = (df_data_raw.loc[~mask_no_mod,
        [
            "sequence",
            "nter",
            "cter",
            "lyn_cyc",
            "ldmix",
            "non_nat",
            "source",
            "non_hem",
            "activity",
        ],
    ]
    .copy().reset_index(drop=True)
)

df_modified["modification"] = df_modified.apply(describe_modifications, axis=1,)
df_modified = df_modified[["sequence", "modification", "source", "non_hem", "activity",]]

In [5]:
pd.Series(
    {
        "raw_records": len(df_data_raw),
        "non_modified_records": len(df_non_modified),
        "modified_records": len(df_modified),
    }
)

raw_records             13215
non_modified_records     4413
modified_records         8802
dtype: int64

- Preparing hemolytic classification labels

In [6]:
df_non_modified["label"] = (df_non_modified["non_hem"].apply(assign_hemolytic_label))
df_modified["label"] = (df_modified["non_hem"].apply(assign_hemolytic_label))
df_non_modified["label"].value_counts(dropna=False).sort_index()

label
0     931
1    3482
Name: count, dtype: int64

- Checking classification duplicates

In [7]:
df_classification_nonmodified = df_non_modified[["sequence", "label"]].copy()
df_remove_duplicated_nonmodified, df_errors_nonmodified, df_unique_nonmodified = processing_duplicated(df_classification_nonmodified, group_seq="sequence", sort_key="label",)
df_full_nonmodified = (pd.concat([df_unique_nonmodified, df_remove_duplicated_nonmodified,],ignore_index=True,).drop_duplicates().reset_index(drop=True))
df_full_nonmodified.shape

(2419, 2)

In [8]:
# Modified peptide identity depends on both the amino-acid sequence and
# its modification annotation. Therefore, modified variants are not collapsed
# solely by sequence.

df_classification_modified = (df_modified[["sequence", "modification", "label"]].drop_duplicates().reset_index(drop=True))
modified_label_counts = (df_classification_modified.groupby(["sequence", "modification"], dropna=False,)["label"].nunique().reset_index(name="number_of_labels"))
modified_conflict_keys = modified_label_counts.loc[modified_label_counts["number_of_labels"] > 1, ["sequence", "modification"],]
df_errors_modified = df_classification_modified.merge(modified_conflict_keys, on=["sequence", "modification"], how="inner",)

df_full_modified = (df_classification_modified.merge(modified_conflict_keys.assign(_conflict=True), on=["sequence", "modification"], how="left",)
    .loc[lambda d: d["_conflict"].isna()]
    .drop(columns="_conflict")
    .drop_duplicates(
        subset=["sequence", "modification", "label"]
    )
    .reset_index(drop=True)
)
df_full_modified.shape

(4689, 3)

- Parsing quantitative hemolytic activity

In [9]:
df_regression_nonmodified, df_regression_errors = prepare_regression_data(
    df_non_modified,
    activity_pattern,
    modified=False,
)

df_regression_modified, df_regression_errors_modified = prepare_regression_data(
    df_modified,
    activity_pattern,
    modified=True,
)

df_regression_nonmodified.shape, df_regression_modified.shape

((169, 5), (468, 6))

In [10]:
df_hc50 = select_endpoint(df_regression_nonmodified, "HC50",)
df_lc50 = select_endpoint(df_regression_nonmodified, "LC50",)
df_ld50 = select_endpoint(df_regression_nonmodified, "LD50",)
df_mhc = select_endpoint(df_regression_nonmodified, "MHC",)
df_hc50_mod = select_endpoint(df_regression_modified, "HC50",)
df_lc50_mod = select_endpoint(df_regression_modified, "LC50",)
df_ld50_mod = select_endpoint(df_regression_modified, "LD50",)
df_mhc_mod = select_endpoint(df_regression_modified, "MHC",)

pd.DataFrame(
    {
        "endpoint": ["HC50", "LC50", "LD50", "MHC"],
        "non_modified": [
            len(df_hc50),
            len(df_lc50),
            len(df_ld50),
            len(df_mhc),
        ],
        "modified": [
            len(df_hc50_mod),
            len(df_lc50_mod),
            len(df_ld50_mod),
            len(df_mhc_mod),
        ],
    }
)

,endpoint,non_modified,modified
0,HC50,44,89
1,LC50,59,119
2,LD50,21,70
3,MHC,45,190


- Working with metadata

In [11]:
df_metadata = read_metadata("../../raw_data/raw_data_description.xlsx", name_source,)
base_metadata = create_metada_with_multiple_values(df_metadata)

In [12]:
classification_metadata = dict(base_metadata)

classification_metadata.update(
    {
        "number_of_raw_sequences": int(number_of_raw_sequences),
        "number_of_sequences_retained": int(len(df_full_nonmodified)),
        "number_of_positive_sequences": int(
            (df_full_nonmodified["label"] == 1).sum()
        ),
        "number_of_negative_sequences": int(
            (df_full_nonmodified["label"] == 0).sum()
        ),
        "number_of_erroneous_sequences": int(
            len(df_errors_nonmodified)
        ),
        "number_of_modified_sequences": int(
            len(df_full_modified)
        ),
        "number_of_erroneous_modified_sequences": int(
            len(df_errors_modified)
        ),
        "modified_sequences_included": False,
    }
)

classification_metadata

{'type source': 'Database',
 'static-dynamic': 'Static',
 'license': 'GNU general public license',
 'year of publication': 2026,
 'last update date': datetime.datetime(2026, 5, 14, 0, 0),
 'download date': Timestamp('2026-09-02 00:00:00'),
 'file format': 'csv',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'Positive, Negative, HC50, LD50, LC50, MHC',
 'unit of measurement': 'µM, mg/L',
 'obtaining negative dataset': 'Experimentally validated',
 'repository or server': 'https://github.com/raghavagps/Hemolytik2/tree/Database',
 'publication': 'https://pubs.acs.org/crtoec/article-abstract/39/2/208/5087119/Hemolytik-2-An-Updated-Database-of-Hemolytic?redirectedFrom=fulltext',
 'number_of_raw_sequences': 13215,
 'number_of_sequences_retained': 2419,
 'number_of_positive_sequences': 1910,
 'number_of_negative_sequences': 509,
 'number_of_erroneous_sequences': 98,
 'number_of_modified_sequences': 4689,
 'number_of_erroneous_modified_sequences': 336,
 'modified_sequences_in

In [13]:
regression_metadata = dict(base_metadata)

regression_metadata.update(
    {
        "number_of_raw_sequences": int(number_of_raw_sequences),
        "number_of_HC50": int(len(df_hc50)),
        "number_of_LC50": int(len(df_lc50)),
        "number_of_LD50": int(len(df_ld50)),
        "number_of_MHC": int(len(df_mhc)),
        "number_of_HC50_mod": int(len(df_hc50_mod)),
        "number_of_LC50_mod": int(len(df_lc50_mod)),
        "number_of_LD50_mod": int(len(df_ld50_mod)),
        "number_of_MHC_mod": int(len(df_mhc_mod)),
        "number_of_unparsed_activity_records": int(
            len(df_regression_errors)
        ),
        "number_of_unparsed_modified_activity_records": int(
            len(df_regression_errors_modified)
        ),
        "modified_sequences_included": False,
    }
)

regression_metadata

{'type source': 'Database',
 'static-dynamic': 'Static',
 'license': 'GNU general public license',
 'year of publication': 2026,
 'last update date': datetime.datetime(2026, 5, 14, 0, 0),
 'download date': Timestamp('2026-09-02 00:00:00'),
 'file format': 'csv',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'Positive, Negative, HC50, LD50, LC50, MHC',
 'unit of measurement': 'µM, mg/L',
 'obtaining negative dataset': 'Experimentally validated',
 'repository or server': 'https://github.com/raghavagps/Hemolytik2/tree/Database',
 'publication': 'https://pubs.acs.org/crtoec/article-abstract/39/2/208/5087119/Hemolytik-2-An-Updated-Database-of-Hemolytic?redirectedFrom=fulltext',
 'number_of_raw_sequences': 13215,
 'number_of_HC50': 44,
 'number_of_LC50': 59,
 'number_of_LD50': 21,
 'number_of_MHC': 45,
 'number_of_HC50_mod': 89,
 'number_of_LC50_mod': 119,
 'number_of_LD50_mod': 70,
 'number_of_MHC_mod': 190,
 'number_of_unparsed_activity_records': 0,
 'number_of_unparsed_

- Exporting classification data

In [14]:
classification_output = (
    f"{PATH_EXPORT}/"
    f"{name_task_classification}/"
    f"{name_source}/"
)

os.makedirs(
    classification_output,
    exist_ok=True,
)

export_json(
    f"{classification_output}/metadata.json",
    classification_metadata,
)

df_full_nonmodified.to_csv(
    f"{classification_output}/processed_hemolytic_dataset.csv",
    index=False,
)

df_full_modified.to_csv(
    f"{classification_output}/modified_hemolytic_dataset.csv",
    index=False,
)

df_errors_nonmodified.to_csv(
    f"{classification_output}/detected_error_sequences.csv",
    index=False,
)

df_errors_modified.to_csv(
    f"{classification_output}/detected_error_modified_sequences.csv",
    index=False,
)

- Exporting regression data

In [15]:
regression_output = (
    f"{PATH_EXPORT}/"
    f"{name_task_regression}/"
    f"{name_source}/"
)

os.makedirs(
    regression_output,
    exist_ok=True,
)

export_json(
    f"{regression_output}/metadata.json",
    regression_metadata,
)

endpoint_datasets = {
    "HC50": df_hc50,
    "LC50": df_lc50,
    "LD50": df_ld50,
    "MHC": df_mhc,
}

modified_endpoint_datasets = {
    "HC50": df_hc50_mod,
    "LC50": df_lc50_mod,
    "LD50": df_ld50_mod,
    "MHC": df_mhc_mod,
}

for endpoint, dataset in endpoint_datasets.items():
    dataset.to_csv(
        f"{regression_output}/processed_{endpoint}_dataset.csv",
        index=False,
    )

for endpoint, dataset in modified_endpoint_datasets.items():
    dataset.to_csv(
        f"{regression_output}/modified_{endpoint}_dataset.csv",
        index=False,
    )

df_regression_errors.to_csv(
    f"{regression_output}/detected_unparsed_activity_records.csv",
    index=False,
)

df_regression_errors_modified.to_csv(
    f"{regression_output}/detected_unparsed_modified_activity_records.csv",
    index=False,
)